**Пример бинарной классификации опухолей молочной железы с использованием LightGBM**

In [10]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

from sales_dataset import SalesDataset

In [11]:
# 1. Загрузка данных
# data = load_breast_cancer()
# X, y = data.data, data.target

df_xls = pd.read_excel('temp_8_9_10_итог_с_июлем.xlsx')
ds = SalesDataset(df_xls, start_date='2024-01-01', cn_mes=19)
X, y = ds.load_data_class()

# 2. Разделение данных
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Создание датасетов LightGBM
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

In [12]:
print(f'{type(X)=}, {X.shape=}')
print(f'{type(y)=}, {y.shape=}')

type(X)=<class 'numpy.ndarray'>, X.shape=(4452, 18)
type(y)=<class 'numpy.ndarray'>, y.shape=(4452,)


In [13]:
# 4. Задание параметров модели
# params = {
#     'objective': 'binary',           # Задача бинарной классификации
#     'metric': 'binary_logloss',      # Метрика оценки - логлосс
#     'boosting_type': 'gbdt',
#     'num_leaves': 31,
#     'learning_rate': 0.05,
#     'feature_fraction': 0.9,
#     'verbose': -1
# }

params = {
    'objective': 'multiclass',
    'num_class': 3,                 # <-- количество классов
    'metric': 'multi_logloss',      # или 'multi_error'
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'verbose': -1
}

In [14]:
# 5. Обучение модели
gbm = lgb.train(params,
                train_data,
                num_boost_round=100,
                valid_sets=[test_data],
                callbacks=[lgb.early_stopping(10)])

Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[30]	valid_0's multi_logloss: 0.802688


In [15]:
print(f'{type(X_test)=}, {X_test.shape=}')

type(X_test)=<class 'numpy.ndarray'>, X_test.shape=(891, 18)


In [16]:
from sklearn.metrics import precision_score, recall_score, f1_score

# 6. Прогнозирование и оценка
# Предсказание вероятностей
y_pred_prob = gbm.predict(X_test, num_iteration=gbm.best_iteration)
# Метки: класс с максимальной вероятностью
y_pred = np.argmax(y_pred_prob, axis=1)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

# Precision/Recall/F1
# macro — среднее по классам (каждый класс одинаково важен)
precision_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
recall_macro    = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1_macro        = f1_score(y_test, y_pred, average='macro', zero_division=0)

# weighted — среднее по классам с весами по support (учитывает дисбаланс)
precision_weighted = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall_weighted    = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1_weighted        = f1_score(y_test, y_pred, average='weighted', zero_division=0)

# ROC-AUC для мультикласса (необязательно)
# Нужно, чтобы y_test были метки 0..K-1, а y_pred_prob (n_samples, K)
# multi_class: 'ovr' или 'ovo'
try:
    auc_ovr_macro = roc_auc_score(y_test, y_pred_prob, multi_class='ovr', average='macro')
except ValueError:
    auc_ovr_macro = None

print(f'Accuracy: {accuracy:.3f}')
if auc_ovr_macro is not None:
    print(f'ROC-AUC (OVR, macro): {auc_ovr_macro:.3f}')

print(f'Precision (macro): {precision_macro:.3f}')
print(f'Recall (macro):    {recall_macro:.3f}')
print(f'F1 (macro):        {f1_macro:.3f}')

print(f'Precision (weighted): {precision_weighted:.3f}')
print(f'Recall (weighted):    {recall_weighted:.3f}')
print(f'F1 (weighted):        {f1_weighted:.3f}')

Accuracy: 0.654
ROC-AUC (OVR, macro): 0.713
Precision (macro): 0.436
Recall (macro):    0.476
F1 (macro):        0.455
Precision (weighted): 0.600
Recall (weighted):    0.654
F1 (weighted):        0.626


**Создание объединённого набора данных**

фичи + прогноз_направление_тренда

In [17]:
df_pred_trend = pd.DataFrame(y_pred, columns=['pred_trend'])
X, y = ds.load_data_regress()
df_x = pd.DataFrame(X)
df_y = pd.DataFrame(y, columns=['fact_sale'])
df_union = pd.concat([df_x, df_y, df_pred_trend], axis=1)

In [18]:
# df_union.round(3)
df_union.round(3).to_csv('features_sale.csv', index=False)